In [1]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [2]:
from datasets import load_dataset
dataset_name = "ridwanFatur98/synth-invoice"
train_dataset = load_dataset(dataset_name, split="train")
sample = train_dataset[0]

In [3]:
from scripts.image_process import process_image, image_to_tensor
from scripts.ground_truth_process import process_ground_truth, get_train_inputs
from scripts.model import DonutModel, DonutConfig
from scripts.tokenizer import get_tokenizer
tokenizer = get_tokenizer()

W0930 18:48:35.539000 16598 site-packages/torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W0930 18:48:35.576000 16598 site-packages/torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


In [4]:
config = DonutConfig(
    input_size=(1280, 960),
    window_size=10,
    encoder_layer=(1, 1, 1, 1),
    max_position_embeddings=768,
    decoder_layer=1,
    vocab_size=len(tokenizer),
    pad_token_id=tokenizer.pad_token_id,
)

In [5]:
donut_model = DonutModel(config).to(device)

state_dict = torch.load(
    "donut_model.pth",
    map_location=device,
)

donut_model.load_state_dict(state_dict)
donut_model.eval()
pass

### Inference

In [6]:
from torch.nn.utils.rnn import pad_sequence
from transformers.file_utils import ModelOutput
from scripts.inference import token2json, inference

In [7]:
# Input Config
input_size = [1280, 960]
align_long_axis = False
random_padding = True

# Input Encoder
img = process_image(sample["image"], input_size, align_long_axis, random_padding)
image_tensors = image_to_tensor(img)
image_tensors = image_tensors.unsqueeze(0)
image_tensors = image_tensors.to(device)

In [8]:
# Input Decoder
prompts = '<s_cord-v2>'
max_length = 768
pad_token_id = tokenizer.pad_token_id
prompt_tensors = tokenizer(
    prompts,
    add_special_tokens=False,
    truncation=True,
    return_tensors="pt",
)["input_ids"].to(device)

In [9]:
prompt_tensors

tensor([[57567]], device='cuda:0')

In [10]:
result = inference(donut_model, tokenizer, image_tensors, prompt_tensors, max_length)

[transformers] The following generation flags are not valid and may be ignored: ['early_stopping']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


In [11]:
result

{'predictions': [{'bill_to': {'name': 'PT Maju Teknologi',
    'address': 'Gg. Tebet Barat Dalam No. 40',
    'city': 'Bontang, Indonesia',
    'phone': '+62 823-9935-2424',
    'email': 'finance@majuteknologi.com'},
   'invoice_date': '1 February 2025',
   'invoice_no': 'INV-2025-30496',
   'from_to': {'name': 'PT Solusi Nusantara',
    'address': 'Gang Antapani Lama No. 34',
    'city': 'Meulaboh, Indonesia',
    'phone': '+62 885-7912-1520',
    'email': 'billing@solusinusantara.com'},
   'items': [{'description': 'DevOps Services',
     'qty': '1',
     'price': '3800000',
     'total': '3800000'},
    {'description': 'UI/UX Design',
     'qty': '9',
     'price': '2900000',
     'total': '26100000'},
    {'description': 'UI/UX Design',
     'qty': '8',
     'price': '4000000',
     'total': '32000000'},
    {'description': 'Cloud Infrastructure',
     'qty': '1',
     'price': '1250000',
     'total': '1250000'},
    {'description': 'AI Development',
     'qty': '7',
     'price':